# E1 — Cross-modal convergence: an image-only encoder vs a text-only encoder

**The question.** Experiments A and B compared models within one modality,
where row *i* was literally the same input. This asks the harder version:
do a vision model that has never seen text and a text model that has never
seen an image build representations related by one linear map? Row *i* here
is an image and *a caption describing it* — two different objects sharing a
referent. This is the Platonic Representation Hypothesis's actual claim.

**Model choice matters.** The image encoder must be self-supervised
(DINOv2, MAE) — **never CLIP-family**, or the text alignment being tested
was trained in, and the experiment is circular. Same for the text side:
a plain text encoder, not a CLIP text tower.

**Pre-registered bands — fixed before any result is seen.**

| Retrieval R@1, as % of the SigLIP ceiling (0.630) | Verdict |
|---|---|
| > 50% | strong cross-modal linear correspondence at this scale |
| 15–50% | measurable but weak — consistent with PRH's scale-dependence |
| 2–15% | marginal; report as a boundary, not a positive result |
| < 2% (chance is 1/1000 = 0.1%) | no linear cross-modal correspondence at this scale |

The middle bands are the honest expectation: PRH predicts alignment grows
with model capability, and these are modest models. A weak-but-above-chance
result is a legitimate finding, not a failure — and repeating with a larger
image encoder would turn this project's two-point sample into a slope.

**Power gate.** The governing ratio is training rows / image-encoder width.
Below 5 the fit is starved and biases *downward* — exactly the run-1 failure
that would fake a negative here. The notebook refuses to interpret results
below that threshold.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU.
#            A free T4 is enough for dinov2-small and -base; -large with
#            cls+patch is happier on an L4. CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU (~4 GB is
#            plenty) or Apple Silicon. Nothing needs changing - the
#            google.colab import below fails harmlessly and the notebook
#            falls through to local mode. Install first:
#            pip install torch torchvision transformers \
#                        sentence-transformers pillow
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On Colab this is
#             the ephemeral VM disk - fast, but everything is LOST when
#             the session ends, so downloads restart from zero.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# checkpoints are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
!pip -q install torch torchvision transformers sentence-transformers pillow

In [ ]:
import numpy as np, torch, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

DATA_DIR = Path(os.environ["DATA_DIR"]); DATA_DIR.mkdir(exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(0)

# ---- models ----
# SWEEP KNOB: after changing this, RESTART the session and run top to
# bottom - N_PAIRS, the checkpoint tag and the power gate all derive
# from it, and a stale IMG in the Colab namespace silently measures the
# wrong encoder. Options: dinov2-small | dinov2-base | dinov2-large.
IMG_MODEL  = "facebook/dinov2-large"    # self-supervised, never saw text
                                        # (a CLIP-family encoder would
                                        #  make this test circular)
TXT_MODEL  = "BAAI/bge-m3"              # text-only (single size only)

# ---- pooling: DINOv2's own linear-probe protocol concatenates the CLS
#      token with the mean of the patch tokens; usually beats CLS alone,
#      but doubles the input width (and so the sample requirement).
POOLING    = "cls+patch"                # "cls" or "cls+patch"

# ---- fair-comparison controls for the scale sweep ----
MATCH_DIM  = None      # e.g. 384: PCA-project the image side to a common
                       # width so a rising trend reflects convergence and
                       # not fit capacity. None = native width.
ALPHA_GRID = [1e-3, 1e-2, 1e-1, 1.0, 10.0]   # tuned on a validation slice

# ---- sample size: must scale with the image encoder's output width ----
# rows/dim >= 5 required; >= 10 comfortable. Widths: small 384, base 768,
# large 1024, giant 1536; x2 if POOLING == "cls+patch".
N_EVAL     = 1000
_BASE_W    = {"small": 384, "base": 768, "large": 1024, "giant": 1536}
_w = _BASE_W[[k for k in _BASE_W if k in IMG_MODEL][0]]
if POOLING == "cls+patch":
    _w *= 2
N_PAIRS    = int(N_EVAL + 10 * _w / 0.9)      # ~10 rows per input dim
CEILING_R1 = 0.630                            # replaced by the 3b cell

print(f"image width will be {_w} ({POOLING}) -> N_PAIRS = {N_PAIRS} "
      f"for ~{(N_PAIRS - N_EVAL) / _w:.1f} rows per input dim")
print("device:", DEV)

## 1. Data — COCO train2017 captions

Uses the official annotation zip (no loader scripts). Images stream from
their URLs; only embeddings are kept, never the pixels.

In [ ]:
import json, zipfile, urllib.request, io

# COCO annotations: cached in DATA_DIR, fetched only if absent or broken.
# ~250 MB zip; only captions_train2017.json is read from it. On the first
# run this is the slowest cell; afterwards it is instant. You can also
# place the file in DATA_DIR yourself and it will be used as-is.
ANN_URL = ("http://images.cocodataset.org/annotations/"
           "annotations_trainval2017.zip")
ANN = DATA_DIR / "annotations_trainval2017.zip"
MEMBER = "annotations/captions_train2017.json"

def _usable(path):
    """exists() is not enough - a truncated download passes it and then
    fails confusingly three cells later."""
    if not path.exists() or path.stat().st_size < 100_000_000:
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return MEMBER in z.namelist()
    except zipfile.BadZipFile:
        return False

if _usable(ANN):
    print(f"using cached annotations ({ANN.stat().st_size/1e6:.0f} MB) "
          f"at {ANN}")
else:
    if ANN.exists():
        print("cached file is truncated or corrupt - re-downloading")
        ANN.unlink()
    print(f"downloading annotations to {ANN} (~250 MB, once) ...")
    tmp = ANN.with_suffix(".part")          # atomic: never leave a half file
    urllib.request.urlretrieve(ANN_URL, str(tmp))
    tmp.rename(ANN)
    assert _usable(ANN), "download completed but the archive is unreadable"
    print(f"done ({ANN.stat().st_size/1e6:.0f} MB)")

with zipfile.ZipFile(ANN) as z:
    with z.open(MEMBER) as f:
        ann = json.load(f)

ALL_CAPTIONS = True        # True: average all 5 caption embeddings per
                           # image (less annotator noise; this is what the
                           # reported results use). False: first caption
                           # only, the standard COCO benchmark protocol.
                           # The 3b ceiling cell matches this setting.

caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
first_cap = {k: (v if ALL_CAPTIONS else v[:1]) for k, v in caps.items()}
url = {im["id"]: im["coco_url"] for im in ann["images"]}
ids = sorted(set(first_cap) & set(url))[:N_PAIRS]
captions = [first_cap[i] for i in ids]   # list of lists
print(f"{len(ids)} image/caption pairs")
print("power check: rows/768 =", f"{0.75*len(ids)/768:.1f}",
      "(need >= 5)")

## 2. Encode — image side (DINOv2), parallel fetch

The images are downloaded from COCO URLs, and that download is the whole
cost of this notebook: sequential fetching leaves the GPU idle >95% of the
time (measured: ~1.5 h for 20k images). This cell fetches with 32 worker
threads, runs the encoder in fp16, and checkpoints every 512 images to
`e1_img_ckpt.npz` — so a Colab disconnect resumes instead of restarting.

Expect a few minutes rather than an hour. Live throughput and ETA are
printed; if a single fetch is slow, raise `WORKERS` to 64.

In [ ]:
import io, urllib.request, numpy as np, torch
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoImageProcessor, AutoModel
from PIL import Image

WORKERS   = 32          # parallel downloads - the whole speedup
CHUNK     = 512         # images fetched per round
BATCH     = 64          # GPU batch
_W_MAP = {"small": 384, "base": 768, "large": 1024, "giant": 1536}
_w = _W_MAP[[k for k in _W_MAP if k in IMG_MODEL][0]] * (
    2 if POOLING == "cls+patch" else 1)
_tag = IMG_MODEL.split("/")[-1] + "_" + POOLING
print(f"encoder {IMG_MODEL}, pooling {POOLING} -> expected width {_w}")
CKPT = DATA_DIR / f"e1_img_ckpt_{_tag}.npz"

proc = AutoImageProcessor.from_pretrained(IMG_MODEL)
vis  = AutoModel.from_pretrained(IMG_MODEL).to(DEV).eval()
if DEV == "cuda":
    vis = vis.half()                      # fp16: ~2x on L4

def fetch(n):
    """returns (row_index, PIL image) or (row_index, None)"""
    try:
        with urllib.request.urlopen(url[ids[n]], timeout=8) as r:
            return n, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return n, None

# resume if a previous run was interrupted
# The checkpoint records which encoder+pooling produced it. Resuming
# across a change mixes embedding widths (base 768 vs large 1024) and
# fails at concatenate - or silently corrupts when widths coincide.
img_list, keep, start = [], [], 0
if CKPT.exists():
    d = np.load(str(CKPT), allow_pickle=True)
    old = str(d["tag"]) if "tag" in d.files else "<unrecorded>"
    got_w = int(d["img"].shape[1]) if "img" in d.files else -1
    if old == _tag and got_w == _w:
        img_list = [d["img"]]; keep = list(d["keep"]); start = int(d["next"])
        print(f"resuming from row {start} ({len(keep)} encoded, {old}, "
              f"width {got_w})")
    else:
        why = (f"tag '{old}' != '{_tag}'" if old != _tag
               else f"width {got_w} != expected {_w}")
        print(f"CHECKPOINT DISCARDED ({why}) - re-encoding from scratch")
        CKPT.unlink()

import time
t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for c0 in range(start, len(ids), CHUNK):
        c1 = min(c0 + CHUNK, len(ids))
        got = sorted((r for r in pool.map(fetch, range(c0, c1))
                      if r[1] is not None), key=lambda x: x[0])
        for b0 in range(0, len(got), BATCH):
            batch = got[b0:b0 + BATCH]
            with torch.no_grad():
                x = proc(images=[im for _, im in batch],
                         return_tensors="pt").to(DEV)
                if DEV == "cuda":
                    x["pixel_values"] = x["pixel_values"].half()
                o = vis(**x).last_hidden_state
                h = (o[:, 0] if POOLING == "cls"
                     else torch.cat([o[:, 0], o[:, 1:].mean(1)], dim=-1))
            img_list.append(h.float().cpu().numpy())
            keep += [n for n, _ in batch]
        done = c1 - start
        rate = done / max(time.time() - t0, 1e-9)
        eta = (len(ids) - c1) / max(rate, 1e-9) / 60
        print(f"  {c1}/{len(ids)}  kept {len(keep)}  "
              f"{rate:.0f} img/s  ETA {eta:.1f} min")
        np.savez_compressed(str(CKPT),
                            img=np.concatenate(img_list).astype(np.float32),
                            keep=np.array(keep), next=c1,
                            tag=np.array(_tag))

IMG = np.concatenate(img_list).astype(np.float64)
assert IMG.shape[1] == _w, (
    f"encoder produced {IMG.shape[1]}-d but {POOLING} on {IMG_MODEL} "
    f"should give {_w} - the pooling branch is not being applied")
print("image embeddings:", IMG.shape,
      f"in {(time.time()-t0)/60:.1f} min")


In [ ]:
from sentence_transformers import SentenceTransformer
txt_model = SentenceTransformer(TXT_MODEL, device=DEV)

# average all captions per image: the target becomes "what the image is
# about" rather than "what one annotator happened to write"
flat, owner = [], []
for j, k in enumerate(keep):
    for cap in captions[k]:
        flat.append(cap); owner.append(j)
E = txt_model.encode(flat, batch_size=128, show_progress_bar=True,
                     convert_to_numpy=True).astype(np.float64)
owner = np.array(owner)
TXT = np.zeros((len(keep), E.shape[1]))
for j in range(len(keep)):
    TXT[j] = E[owner == j].mean(0)
print(f"{len(flat)} captions -> {TXT.shape} averaged targets "
      f"({len(flat)/len(keep):.1f} per image)")

assert len(IMG) == len(TXT), "row alignment broken"
np.savez_compressed(str(DATA_DIR / "crossmodal_pairs.npz"),
                    img=IMG.astype(np.float32), txt=TXT.astype(np.float32))
print("saved crossmodal_pairs.npz")

## 3. The power gate — refuse to interpret an underpowered result

In [ ]:
# CONSISTENCY GUARD - catches stale variables and mis-set pooling.
# Colab keeps state between runs: if IMG is left over from an earlier
# encoder, everything below silently measures the wrong model.
_expect_w = _w                      # from the config cell
_expect_n = N_PAIRS
print(f"config expects: width {_expect_w}, up to {_expect_n} rows "
      f"({IMG_MODEL}, {POOLING})")
print(f"IMG in memory : width {IMG.shape[1]}, {IMG.shape[0]} rows")
assert IMG.shape[1] == _expect_w, (
    f"WIDTH MISMATCH: IMG is {IMG.shape[1]}-d but {IMG_MODEL} with "
    f"POOLING={POOLING} should give {_expect_w}. Either the encode cell "
    f"did not re-run after changing the config, or the pooling branch "
    f"is not being applied. Re-run the encode cell.")
assert IMG.shape[0] == TXT.shape[0], "IMG/TXT row mismatch"
assert IMG.shape[0] > 0.9 * _expect_n, (
    f"ROW COUNT MISMATCH: {IMG.shape[0]} rows but the config asked for "
    f"{_expect_n} - IMG is probably left over from a previous run.")
print("consistency OK\n")

# optional: project the image side to a common width so that a scale sweep
# compares convergence rather than fit capacity
if MATCH_DIM and MATCH_DIM < IMG.shape[1]:
    Xc = IMG - IMG.mean(0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    IMG_FIT = Xc @ Vt[:MATCH_DIM].T
    print(f"PCA-matched image side: {IMG.shape[1]} -> {MATCH_DIM} dims")
else:
    IMG_FIT = IMG
    print(f"native image width: {IMG.shape[1]} dims")

d_img = IMG_FIT.shape[1]
idx = rng.permutation(len(IMG_FIT))
te, tr = idx[:N_EVAL], idx[N_EVAL:]
ratio = len(tr) / d_img
print(f"training rows {len(tr)} / image dim {d_img} = {ratio:.1f}")
POWERED = ratio >= 5
print("POWERED" if POWERED else
      "UNDERPOWERED - a negative result here is NOT interpretable; "
      "increase N_PAIRS before drawing any conclusion")

In [ ]:
# shared helpers - defined once, used by the ceiling cell and the evaluation
def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def ridge(X, Y, a=1e-2):
    """alpha is selected per run by the ALPHA_GRID sweep in the
    evaluation cell; this default is only a fallback."""
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

def recall(S):
    order = np.argsort(-S, axis=1)
    r = (order == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

print("helpers ready: l2n, ridge, recall")

## 3b. Measure the ceiling for THIS direction — and on the SAME task

`CEILING_R1 = 0.630` was measured in Experiment B for **text -> image**.
E1 runs **image -> text**, so the borrowed figure is only approximate.

Two things must match between the system and its ceiling, or the
percentage is meaningless:

1. **Direction** — both must be image -> text.
2. **Caption treatment** — if the system retrieves against *averaged*
   caption embeddings (`ALL_CAPTIONS = True`), the ceiling must too.
   Averaging produces a cleaner, more canonical target per image, which
   makes retrieval easier; scoring the system on averaged captions against
   a ceiling measured on single captions inflates the percentage.

This cell measures SigLIP's own two towers on the same held-out rows, with
the same caption treatment.

In [ ]:
# exact ceiling: SigLIP image tower -> SigLIP text tower, same eval rows,
# and the SAME caption treatment as the system (averaged when ALL_CAPTIONS)
from transformers import AutoProcessor, AutoModel as HFAutoModel
from concurrent.futures import ThreadPoolExecutor

SIG = "google/siglip-base-patch16-224"
sp = AutoProcessor.from_pretrained(SIG)
sm = HFAutoModel.from_pretrained(SIG).to(DEV).eval()

def _tensor(o):
    """transformers versions differ: tensor or output object."""
    if torch.is_tensor(o):
        return o
    for a in ("image_embeds", "text_embeds", "pooler_output"):
        v = getattr(o, a, None)
        if v is not None:
            return v
    v = getattr(o, "last_hidden_state", None)
    if v is not None:
        return v.mean(1)
    raise TypeError(f"cannot extract a tensor from {type(o)}")

def _img(n):
    r = fetch(n)
    return r[1] if isinstance(r, tuple) else r

def _caps(n):
    """ALL captions for image n - must match how TXT was built."""
    c = captions[n]
    return c if isinstance(c, list) else [c]

def _sig_text(strings, bs=64):
    out = []
    for b in range(0, len(strings), bs):
        with torch.no_grad():
            y = sp(text=strings[b:b + bs], padding="max_length",
                   truncation=True, return_tensors="pt").to(DEV)
            try:
                v = sm.get_text_features(**y)
            except Exception:
                v = sm(input_ids=y["input_ids"]).text_embeds
            out.append(_tensor(v).float().cpu().numpy())
    return np.concatenate(out)

def _sig_image(images, bs=32):
    out = []
    for b in range(0, len(images), bs):
        with torch.no_grad():
            x = sp(images=images[b:b + bs], return_tensors="pt").to(DEV)
            try:
                v = sm.get_image_features(**x)
            except Exception:
                v = sm(pixel_values=x["pixel_values"]).image_embeds
            out.append(_tensor(v).float().cpu().numpy())
    return np.concatenate(out)

def siglip_ceiling(rows):
    want = [int(keep[r]) for r in rows]
    with ThreadPoolExecutor(max_workers=32) as pool:
        got = list(pool.map(_img, want))
    ok = [(n, im) for n, im in zip(want, got) if im is not None]
    if len(ok) < 100:
        print(f"only {len(ok)} images available - ceiling not measured")
        return None, 0, 0

    I = l2n(_sig_image([im for _, im in ok]))

    # text side, matching the system's treatment exactly
    flat, owner = [], []
    for j, (n, _) in enumerate(ok):
        for cap in _caps(n):
            flat.append(cap); owner.append(j)
    owner = np.array(owner)
    E = _sig_text(flat)
    T = np.stack([E[owner == j].mean(0) for j in range(len(ok))])
    T = l2n(T)
    per_img = len(flat) / len(ok)
    return recall(I @ T.T), len(ok), per_img

ceil, n_gal, per_img = siglip_ceiling(te)
if ceil:
    CEILING_R1 = ceil[1]
    print(f"MEASURED image->text ceiling (SigLIP, gallery {n_gal}, "
          f"{per_img:.1f} captions/image averaged):")
    print("   " + "  ".join(f"R@{k}={v:.3f}" for k, v in ceil.items()))
    print("   caption treatment now MATCHES the system - the system and")
    print("   the ceiling are scored on the same task")
    print("   (Experiment B's borrowed text->image figure was 0.630;")
    print("    a single-caption ceiling measured earlier gave 0.648)")
    if n_gal < len(te):
        print(f"   note: gallery {n_gal}, not {len(te)} - R@K depends on")
        print("   gallery size, compare only against this count")
else:
    print("keeping the borrowed ceiling 0.630")

## 4. Fit and evaluate, with all four controls

1. **shuffle control** — misaligned rows; the honest fit must beat it
2. **random-weights image encoder** — architecture-only baseline
3. **raw cross-space** — the chance floor
4. **same-modality reference** — this project's own 0.592 / 0.737

In [ ]:
# tune alpha on a validation slice carved out of TRAIN only
nv = max(200, len(tr) // 5)
vtr, vva = tr[:-nv], tr[-nv:]
best = (None, -9)
for a in ALPHA_GRID:
    Wa = ridge(IMG_FIT[vtr], TXT[vtr], a)
    Pv = IMG_FIT[vva] @ Wa
    sc = 1 - ((TXT[vva] - Pv) ** 2).sum() / \
             ((TXT[vva] - TXT[vva].mean(0)) ** 2).sum()
    print(f"  alpha {a:<7g} validation R2 {sc:.3f}")
    if sc > best[1]:
        best = (a, sc)
ALPHA_BEST = best[0]
print(f"  -> alpha = {ALPHA_BEST} (validation R2 {best[1]:.3f})\n")

W = ridge(IMG_FIT[tr], TXT[tr], ALPHA_BEST)
P = IMG_FIT[te] @ W
r2 = 1 - ((TXT[te]-P)**2).sum() / ((TXT[te]-TXT[te].mean(0))**2).sum()
cos = float((l2n(P) * l2n(TXT[te])).sum(1).mean())

Ws = ridge(IMG_FIT[tr], TXT[tr][rng.permutation(len(tr))],
           ALPHA_BEST)
r2s = 1 - ((TXT[te]-IMG_FIT[te]@Ws)**2).sum() / \
          ((TXT[te]-TXT[te].mean(0))**2).sum()

gal = l2n(TXT[te])
r_adapt = recall(l2n(P) @ gal.T)
raw = np.zeros_like(TXT[te]); m = min(d_img, TXT.shape[1])
raw[:, :m] = IMG_FIT[te][:, :m]
r_raw = recall(l2n(raw) @ gal.T)

print(f"held-out ridge : R2 {r2:.3f}   cosine {cos:.3f}")
print(f"shuffle control: R2 {r2s:.3f}   (gap must exceed 0.2)")
print(f"retrieval image->text: " +
      "  ".join(f"R@{k}={v:.3f}" for k, v in r_adapt.items()))
print(f"raw (chance floor)   : " +
      "  ".join(f"R@{k}={v:.3f}" for k, v in r_raw.items()))
pct = 100 * r_adapt[1] / CEILING_R1
print(f"\nR@1 as % of the SigLIP ceiling ({CEILING_R1}): {pct:.1f}%")
print(f"same-modality reference from this project: "
      f"R2 0.592 (vision pair), 0.737 (LLM pair)")

if not POWERED:
    print("\nVERDICT WITHHELD - underpowered")
elif r2 - r2s < 0.2:
    print("\nVERDICT: alignment check FAILED - fit does not beat the "
          "shuffle control; suspect row misalignment before anything else")
elif pct > 50:
    print("\nVERDICT: STRONG cross-modal linear correspondence")
elif pct > 15:
    print("\nVERDICT: MEASURABLE BUT WEAK - consistent with PRH's "
          "scale-dependence. Repeat with a larger image encoder to get a "
          "slope rather than a point.")
elif pct > 2:
    print("\nVERDICT: MARGINAL - report as a boundary, not a positive")
else:
    print("\nVERDICT: NO linear cross-modal correspondence at this scale")

## 5. Optional — the scale slope (the point of the whole exercise)

Re-run cells 2–4 with a larger image encoder (`facebook/dinov2-large`,
then `dinov2-giant`) keeping everything else fixed, and plot R@1 against
encoder size. PRH predicts the trend rises. Two points make a line; three
make it credible — and a slope is exactly what this project's stated
limitation ('two small models are a two-point sample') asks for.

In [ ]:
# ---- scale sweep driver -------------------------------------------------
# Re-run this notebook once per encoder, changing ONLY IMG_MODEL, then
# record the numbers here. N_PAIRS auto-scales with width; set MATCH_DIM
# to the smallest width in the sweep for the capacity-controlled variant.
#
#   IMG_MODEL = "facebook/dinov2-small"   # width 384  (768 with cls+patch)
#   IMG_MODEL = "facebook/dinov2-base"    # width 768  (1536)
#   IMG_MODEL = "facebook/dinov2-large"   # width 1024 (2048)
#
import matplotlib.pyplot as plt

results = {            # fill in as each run completes
    # "small": dict(params=22,  r1=None, r2=None),
    # "base":  dict(params=87,  r1=0.358, r2=0.511),
    # "large": dict(params=304, r1=None, r2=None),
}
done = {k: v for k, v in results.items() if v.get("r1") is not None}
if len(done) >= 2:
    ks = list(done); x = [done[k]["params"] for k in ks]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, key, lab in [(ax[0], "r1", "R@1"), (ax[1], "r2", "held-out R2")]:
        a.plot(x, [done[k][key] for k in ks], "o-", lw=2)
        for k, xx in zip(ks, x):
            a.annotate(k, (xx, done[k][key]), textcoords="offset points",
                       xytext=(4, 6), fontsize=8)
        a.set_xscale("log"); a.set_xlabel("image encoder params (M)")
        a.set_ylabel(lab); a.grid(alpha=0.3)
    plt.suptitle("Cross-modal alignment vs image-encoder capacity")
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "e1_scale_slope.png"), dpi=150); plt.show()
    print("rising = evidence for scale-dependent convergence (PRH);")
    print("flat  = an equally reportable negative at these scales")
else:
    print(f"{len(done)} of {len(results)} runs recorded - need at least 2")